<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Table of Contents:

- [Imports](#imports)
- [Notebook Notes](#notebook-notes)
- [Interactive Control Setup](#interactive-control-setup)
- [Simulation Output Helpers](#simulation-output-helpers)
- [Run Interactive Simulation](#run-interactive-simulation)
- [Preset Scenario Examples](#preset-scenario-examples)
- [Final Simulation Discussion](#final-simulation-discussion)

</div>

##### **IMPORTANT**:
- This notebook is a text-based RF simulation, not a production radio receiver.
- The project Random Forest models classify modulation type from I/Q-style signal data; they do **not** decrypt messages.
- Modulation and encryption are separate layers. A receiver needs the correct demodulation approach and the correct decryption key to recover the original text.
- The generated I/Q samples are shaped into a DeepSig-style array format for consistency with the project dataset.

---

##### **ORIENTATION**:
- The user inputs text.
- The notebook converts the text into bits.
- Optional XOR encryption masks the original bits.
- The transmitter modulates the bits into I/Q samples.
- The receiver can intentionally use the correct or wrong modulation assumption.
- The receiver can intentionally use the correct or wrong encryption key.
- The notebook shows the recovered bits and output text for each case.

---

##### **SUMMARY**:
- Correct modulation + correct encryption should recover the original text.
- Wrong modulation corrupts the recovered bitstream before decryption.
- Wrong encryption corrupts the recovered text even when modulation is correct.
- This simulation supports the final project discussion by making the bit-level communication pipeline visible.

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Imports

- [Back to Table of Contents](#table-of-contents)

</div>

This section imports the simulation helpers and interactive notebook tools.

The path logic allows the notebook to be run either from inside the `Simulation/` folder or from the project root.

In [1]:
import sys
from pathlib import Path

# Allow notebook to import from the Simulation folder
simulation_dir = Path.cwd()

if simulation_dir.name != "Simulation":
    simulation_dir = Path.cwd() / "Simulation"

if str(simulation_dir) not in sys.path:
    sys.path.append(str(simulation_dir))

from simulation_core import (
    AVAILABLE_MODULATION_TYPES,
    run_text_simulation,
    format_bits
)

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Install with: pip install ipywidgets")
    print(exc)

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Notebook Notes

- [Back to Table of Contents](#table-of-contents)

</div>

This notebook demonstrates the communication pipeline used to explain the highest-SNR modeling project.

The simplified pipeline is:

```text
text
→ true original bits
→ optional encryption
→ transmitted bits
→ modulation
→ I/Q samples
→ DeepSig-style X frames
→ receiver demodulation
→ optional decryption
→ recovered bits
→ recovered text
```

The simulation is intentionally text-based because text makes the bit-level effects easier to interpret than audio. Audio can be added later, but text is better for explaining how wrong modulation and wrong encryption affect recovery.

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Interactive Control Setup

- [Back to Table of Contents](#table-of-contents)

</div>

This section creates the interactive controls.

Use these controls to change the transmitter setup and receiver assumptions:

```text
True modulation:
    The modulation used by the transmitter.

Correct modulation enabled:
    If checked, the receiver uses the true modulation.
    If unchecked, the receiver uses the selected receiver modulation.

Encryption enabled:
    If checked, the original bits are XOR-masked before modulation.

Correct encryption enabled:
    If checked, the receiver uses the true encryption key.
    If unchecked, the receiver uses the wrong key.
```

In [ ]:
if WIDGETS_AVAILABLE:
    input_text_widget = widgets.Textarea(
        value="HELLO RF WORLD",
        description="Input text:",
        layout=widgets.Layout(width="95%", height="80px")
    )

    true_modulation_widget = widgets.Dropdown(
        options=AVAILABLE_MODULATION_TYPES,
        value="QPSK",
        description="True mod:"
    )

    correct_modulation_widget = widgets.Checkbox(
        value=True,
        description="Correct modulation enabled"
    )

    receiver_modulation_widget = widgets.Dropdown(
        options=AVAILABLE_MODULATION_TYPES,
        value="BPSK",
        description="Receiver mod:"
    )

    encryption_enabled_widget = widgets.Checkbox(
        value=True,
        description="Encryption enabled"
    )

    encryption_key_widget = widgets.Text(
        value="secret",
        description="True key:"
    )

    correct_encryption_widget = widgets.Checkbox(
        value=True,
        description="Correct encryption enabled"
    )

    receiver_key_widget = widgets.Text(
        value="wrong_key",
        description="Receiver key:"
    )

    show_full_keys_widget = widgets.Checkbox(
        value=False,
        description="Reveal keys in output"
    )

    run_button = widgets.Button(
        description="Run Simulation",
        button_style="success"
    )

    output_area = widgets.Output()

    controls_left = widgets.VBox([
        input_text_widget,
        true_modulation_widget,
        correct_modulation_widget,
        receiver_modulation_widget
    ])

    controls_right = widgets.VBox([
        encryption_enabled_widget,
        encryption_key_widget,
        correct_encryption_widget,
        receiver_key_widget,
        show_full_keys_widget,
        run_button
    ])

    display(widgets.HBox([controls_left, controls_right]))
    display(output_area)

else:
    print("Widgets unavailable. Install ipywidgets to use the interactive session.")

Output()

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Simulation Output Helpers

- [Back to Table of Contents](#table-of-contents)

</div>

This section defines display helpers and the button-click behavior.

The output is organized around the major communication stages:

```text
1. True plaintext
2. Modulation masking / receiver assumption
3. Encryption masking / receiver assumption
4. Output text
5. DeepSig-style I/Q format
6. I/Q constellation preview
```

In [3]:
def render_bits_section(title, bits_display):
    display(Markdown(f"### {title}"))
    display(Markdown(f"`{bits_display}`"))


def plot_iq_constellation(iq_samples, title="I/Q Constellation Preview", max_points=512):
    shown = iq_samples[:max_points]

    plt.figure(figsize=(7, 5))
    plt.scatter(shown.real, shown.imag, s=10, label="I/Q samples")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.grid(True)
    plt.xlabel("I")
    plt.ylabel("Q")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


def on_run_button_clicked(button):
    with output_area:
        clear_output(wait=True)

        results = run_text_simulation(
            input_text=input_text_widget.value,
            true_modulation_type=true_modulation_widget.value,
            receiver_modulation_type=receiver_modulation_widget.value,
            correct_modulation_enabled=correct_modulation_widget.value,
            encryption_enabled=encryption_enabled_widget.value,
            encryption_key=encryption_key_widget.value,
            receiver_encryption_key=receiver_key_widget.value,
            correct_encryption_enabled=correct_encryption_widget.value
        )

        display(Markdown("# Simulation Results"))

        display(Markdown("## TRUE PLAINTEXT"))
        display(Markdown(f"**Input text:** `{results['input_text']}`"))
        display(Markdown(f"**Plain text roundtrip:** `{results['plain_text_roundtrip']}`"))
        render_bits_section("True original bits", results["original_bits_display"])

        display(Markdown("## MODULATION MASKING / RECEIVER ASSUMPTION"))
        display(Markdown(f"**True transmitter modulation:** `{results['true_modulation_type']}`"))
        display(Markdown(f"**Receiver-assumed modulation:** `{results['assumed_modulation_type']}`"))
        display(Markdown(f"**Correct modulation enabled:** `{results['correct_modulation_enabled']}`"))
        render_bits_section("Recovered transmitted bits after demodulation", results["recovered_transmitted_bits_display"])

        display(Markdown("## ENCRYPTION MASKING / RECEIVER ASSUMPTION"))
        display(Markdown(f"**Encryption enabled:** `{results['encryption_enabled']}`"))

        if show_full_keys_widget.value:
            display(Markdown(f"**True encryption key:** `{results['true_encryption_key']}`"))
            display(Markdown(f"**Receiver encryption key:** `{results['assumed_encryption_key']}`"))
        else:
            display(Markdown("**True encryption key:** `[masked]`"))
            display(Markdown("**Receiver encryption key:** `[masked]`"))

        display(Markdown(f"**Correct encryption enabled:** `{results['correct_encryption_enabled']}`"))
        render_bits_section("Transmitted bits after optional encryption", results["transmitted_bits_display"])
        render_bits_section("Recovered original bits after optional decryption", results["recovered_original_bits_display"])

        display(Markdown("## OUTPUT TEXT"))
        display(Markdown(f"**Recovered text:** `{results['recovered_text']}`"))
        display(Markdown(f"**Successful recovery:** `{results['successful_recovery']}`"))
        display(Markdown(f"**Transmitted bit error rate:** `{results['transmitted_bit_error_rate']:.4f}`"))
        display(Markdown(f"**Original bit error rate:** `{results['original_bit_error_rate']:.4f}`"))

        display(Markdown("## DEEPSIG-STYLE I/Q FORMAT"))
        display(Markdown(f"**Raw I/Q sample shape:** `{results['iq_shape']}`"))
        display(Markdown(f"**DeepSig-style X shape:** `{results['deepsig_X_shape']}`"))
        display(Markdown("This corresponds to `X[:, :, 0] = I` and `X[:, :, 1] = Q`."))

        plot_iq_constellation(
            iq_samples=results["iq_samples"],
            title=f"I/Q Preview | True={results['true_modulation_type']} | Receiver={results['assumed_modulation_type']}"
        )


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Run Interactive Simulation

- [Back to Table of Contents](#table-of-contents)

</div>

Run the cell below to activate the interactive simulation.

After the widgets appear, change any value and click **Run Simulation**.

In [4]:
if WIDGETS_AVAILABLE:
    run_button.on_click(on_run_button_clicked)
    on_run_button_clicked(None)

else:
    print("Widgets unavailable. Install with: pip install ipywidgets")

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Preset Scenario Examples

- [Back to Table of Contents](#table-of-contents)

</div>

This section runs fixed scenario examples using the same message, true modulation, and true key.

The examples are useful for a report or presentation because they show the four core cases:

```text
1. Correct modulation + correct encryption
2. Wrong modulation + correct encryption
3. Correct modulation + wrong encryption
4. Wrong modulation + wrong encryption
```

In [5]:
message = "HELLO RF WORLD"
true_mod = "QPSK"
true_key = "secret"

scenarios = [
    {
        "name": "Correct modulation + correct encryption",
        "correct_modulation_enabled": True,
        "correct_encryption_enabled": True
    },
    {
        "name": "Wrong modulation + correct encryption",
        "correct_modulation_enabled": False,
        "correct_encryption_enabled": True
    },
    {
        "name": "Correct modulation + wrong encryption",
        "correct_modulation_enabled": True,
        "correct_encryption_enabled": False
    },
    {
        "name": "Wrong modulation + wrong encryption",
        "correct_modulation_enabled": False,
        "correct_encryption_enabled": False
    }
]

scenario_rows = []

for scenario in scenarios:
    results = run_text_simulation(
        input_text=message,
        true_modulation_type=true_mod,
        receiver_modulation_type="BPSK",
        correct_modulation_enabled=scenario["correct_modulation_enabled"],
        encryption_enabled=True,
        encryption_key=true_key,
        receiver_encryption_key="wrong_key",
        correct_encryption_enabled=scenario["correct_encryption_enabled"]
    )

    scenario_rows.append({
        "scenario": scenario["name"],
        "true_modulation": results["true_modulation_type"],
        "receiver_modulation": results["assumed_modulation_type"],
        "correct_modulation": results["correct_modulation_enabled"],
        "correct_encryption": results["correct_encryption_enabled"],
        "successful_recovery": results["successful_recovery"],
        "original_bit_error_rate": results["original_bit_error_rate"],
        "recovered_text": results["recovered_text"]
    })

import pandas as pd

scenario_df = pd.DataFrame(scenario_rows)
scenario_df

,scenario,true_modulation,receiver_modulation,correct_modulation,correct_encryption,successful_recovery,original_bit_error_rate,recovered_text
0,Correct modulation + correct encryption,QPSK,QPSK,True,True,True,0.000000,HELLO RF WORLD
1,Wrong modulation + correct encryption,QPSK,BPSK,False,True,False,0.473214,ܬ��\t��ecretse
2,Correct modulation + wrong encryption,QPSK,QPSK,True,False,False,0.312500,LR@PM JF:RXIQF
3,Wrong modulation + wrong encryption,QPSK,BPSK,False,False,False,0.446429,ػ�� ��eywrong


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Final Simulation Discussion

- [Back to Table of Contents](#table-of-contents)

</div>

##### **SUMMARY**:
- The trained RFC models identify modulation-like patterns; encryption remains a separate bitstream operation.
- If the receiver uses the wrong modulation, the recovered transmitted bits become unreliable.
- If the receiver uses the wrong encryption key, the demodulated bits may be correct but the final text is still unreadable.
- The successful path requires both correct modulation recovery and correct encryption recovery.
- This provides a clean final demonstration connecting modulation classification, signal interpretation, and bit-level communication.